##### PHASE 1 - DATA TRANSFER FROM SOURCE FILES TO DESTINATION FILES (MULTIPLE SOURCE FILES ARE GIVEN AS AN INPUT WITH PRIORITY)

##### PHASE 2 - EXTRACTING THE FUNCTION NAME AND VERSION OF THE LABELS THAT NEED ATTENTION

In [9]:
"""
A2L Parser - Grouped Layout Output
Layout per group:
  Row 1 : [function_full_name] [function_name] [function_version] [label=HEADER MARKER]
  Row 2+: [function_full_name] [function_name] [function_version] [label]  <- repeated for all labels
  Row N+1, N+2: blank spacer rows
  ... next function group ...
"""

import re
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

# ─── SET YOUR PATHS HERE ─────────────────────────────────────────────────────
excel_path  = r"C:\Users\SHUE1KOR\Desktop\CalAi\A2L_HEX_parser\needs_attention_labels.xlsx"
a2l_path    = r"C:\Users\SHUE1KOR\Desktop\CalAi\A2L_HEX_parser\Dest\MD1CV_P2572_MD1CE300_2_0_0.a2l"
output_path = excel_path   # Overwrites same file; change if you want a separate output
# ─────────────────────────────────────────────────────────────────────────────


def build_label_lookup(a2l_path: str):
    """Single-pass parse. Returns label->func_name and label->func_version dicts."""
    label_to_func = {}
    label_to_ver  = {}

    func_block_re = re.compile(r'/begin\s+FUNCTION',          re.IGNORECASE)
    end_func_re   = re.compile(r'/end\s+FUNCTION',            re.IGNORECASE)
    def_char_re   = re.compile(r'/begin\s+DEF_CHARACTERISTIC', re.IGNORECASE)
    end_def_re    = re.compile(r'/end\s+DEF_CHARACTERISTIC',   re.IGNORECASE)
    func_ver_re   = re.compile(r'FUNCTION_VERSION\s+"([^"]*)"', re.IGNORECASE)

    with open(a2l_path, 'r', encoding='utf-8', errors='replace') as f:
        content = f.read()

    func_starts = [m.start() for m in func_block_re.finditer(content)]
    func_ends   = [m.end()   for m in end_func_re.finditer(content)]

    pairs, ei = [], 0
    for si in func_starts:
        while ei < len(func_ends) and func_ends[ei] <= si:
            ei += 1
        if ei < len(func_ends):
            pairs.append((si, func_ends[ei]))
            ei += 1

    for start, end in pairs:
        block = content[start:end]
        lines = block.splitlines()
        func_name = None
        for i, line in enumerate(lines):
            if func_block_re.search(line.strip()):
                for j in range(i + 1, len(lines)):
                    c = lines[j].strip()
                    if c and not c.startswith('"') and not c.startswith('/'):
                        func_name = c.split()[0]
                        break
                break
        if func_name is None:
            continue

        ver_match = func_ver_re.search(block)
        func_ver  = ver_match.group(1) if ver_match else ""

        def_match = def_char_re.search(block)
        end_match = end_def_re.search(block)
        if def_match and end_match and def_match.start() < end_match.start():
            for lbl in block[def_match.end(): end_match.start()].split():
                label_to_func[lbl] = func_name
                label_to_ver[lbl]  = func_ver

    print(f"Parsed {len(pairs)} FUNCTION blocks, {len(label_to_func)} unique labels indexed.")
    return label_to_func, label_to_ver


def build_grouped_rows(df, label_col, label_to_func, label_to_ver):
    """
    Groups labels by (function_name, function_version).
    Returns list of row dicts for the output sheet.
    Layout per group:
      - Header row  : function info + label = "--- <func_full_name> ---"
      - Label rows  : function info repeated + each label
      - 2 blank rows
    """
    df['_func']  = df[label_col].map(label_to_func).fillna("NOT FOUND")
    df['_ver']   = df[label_col].map(label_to_ver).fillna("")
    df['_full']  = df['_func'] + '  ' + df['_ver']

    rows = []
    not_found_labels = []

    # Group preserving order of first appearance
    seen_groups = {}
    for _, row in df.iterrows():
        key = (row['_func'], row['_ver'], row['_full'])
        if key not in seen_groups:
            seen_groups[key] = []
        seen_groups[key].append(row[label_col])

    for (func_name, func_ver, func_full), labels in seen_groups.items():
        if func_name == "NOT FOUND":
            not_found_labels.extend(labels)
            continue

        # Group header row — function info shown ONCE here, blank on label rows below
        rows.append({
            'function_full_name': func_full,
            'function_name':      func_name,
            'function_version':   func_ver,
            'label':              f'*** {func_full} ***',
            '_is_header': True
        })
        # One row per label — function columns left BLANK (shown only on header row above)
        for lbl in labels:
            rows.append({
                'function_full_name': '',
                'function_name':      '',
                'function_version':   '',
                'label':              lbl,
                '_is_header': False
            })
        # 2 blank spacer rows
        rows.append({'function_full_name': '', 'function_name': '', 'function_version': '', 'label': '', '_is_header': False})
        rows.append({'function_full_name': '', 'function_name': '', 'function_version': '', 'label': '', '_is_header': False})

    # Append NOT FOUND labels at the end as their own group
    if not_found_labels:
        rows.append({
            'function_full_name': 'NOT FOUND',
            'function_name':      'NOT FOUND',
            'function_version':   '',
            'label':              '*** NOT FOUND IN A2L ***',
            '_is_header': True
        })
        for lbl in not_found_labels:
            rows.append({
                'function_full_name': 'NOT FOUND',
                'function_name':      'NOT FOUND',
                'function_version':   '',
                'label':              lbl,
                '_is_header': False
            })

    missing = len(not_found_labels)
    print(f"Count of labels NOT FOUND in the A2L : {missing}")
    return rows


def write_excel(rows, output_path):
    cols = ['function_full_name', 'function_name', 'function_version', 'label']
    data = [{c: r[c] for c in cols} for r in rows]
    df_out = pd.DataFrame(data, columns=cols)
    df_out.to_excel(output_path, index=False)

    wb = load_workbook(output_path)
    ws = wb.active

    # Styles
    hdr_font   = Font(name='Arial', bold=True, color='FFFFFF')
    hdr_fill   = PatternFill('solid', start_color='1F4E79')   # dark blue
    grp_font   = Font(name='Arial', bold=True, color='FFFFFF')
    grp_fill   = PatternFill('solid', start_color='2E75B6')   # medium blue
    lbl_font   = Font(name='Arial', size=10)
    center     = Alignment(horizontal='center', vertical='center')
    left       = Alignment(horizontal='left',   vertical='center')
    thin_side  = Side(style='thin', color='AAAAAA')
    thin_border= Border(bottom=thin_side)

    # Format header row (row 1)
    for cell in ws[1]:
        cell.font      = hdr_font
        cell.fill      = hdr_fill
        cell.alignment = center

    # Map row index to _is_header flag
    is_header_map = {i + 2: r['_is_header'] for i, r in enumerate(rows)}  # +2 because row 1 = col headers

    for row_idx, row in enumerate(ws.iter_rows(min_row=2), start=2):
        is_grp_hdr = is_header_map.get(row_idx, False)
        for cell in row:
            if is_grp_hdr:
                cell.font      = grp_font
                cell.fill      = grp_fill
                cell.alignment = center
            else:
                cell.font      = lbl_font
                cell.alignment = left

    # Column widths
    col_widths = {'A': 45, 'B': 25, 'C': 25, 'D': 40}
    for col_letter, width in col_widths.items():
        ws.column_dimensions[col_letter].width = width

    # Freeze top row
    ws.freeze_panes = 'A2'

    wb.save(output_path)
    print(f"Saved → {output_path}")


def main():
    label_to_func, label_to_ver = build_label_lookup(a2l_path)
    df = pd.read_excel(excel_path)
    label_col = next((c for c in df.columns if c.strip().lower() == 'label'), df.columns[0])
    rows = build_grouped_rows(df, label_col, label_to_func, label_to_ver)
    write_excel(rows, output_path)


main()

Parsed 2006 FUNCTION blocks, 87316 unique labels indexed.
Count of labels NOT FOUND in the A2L : 0
Saved → C:\Users\SHUE1KOR\Desktop\CalAi\A2L_HEX_parser\needs_attention_labels.xlsx


In [ ]:
"""

==========================================


Supported label suffixes
------------------------
  _C   / _CW  / _c          Scalar value (single number/string)
  _CA                       1-D array (no axis)
  _MAP / _M                 2-D map  (x-axis + y-axis + grid)
  _T   / _CUR / _Cur        1-D curve (z-values + x-axis)

Anything else is logged under "Unhandled suffix" and left untouched.

Key correctness properties
--------------------------
* Destination dimensions are NEVER changed — only the text of existing
  V / VT / LABEL elements is overwritten.

* _T  uses real physical-coordinate interp/extrap.  Destination x-axis is
  left untouched so (x, z) pairs in the destination remain self-consistent.

* _MAP uses real physical-coordinate bilinear interp/extrap when both axes
  are numeric.  When an axis is non-numeric (enum), that dimension switches
  to label-matched lookup with nearest-neighbour fallback.

* _CA uses index-based resampling (no axis exists → no real interp/extrap
  possible by construction; this is the correct behaviour for axisless
  arrays).

* Floats round-trip through f"{v}".  No "Updated" record is emitted when
  the destination values already match the source — keeps diffs clean.

* Source x-axes are auto-sorted/de-duplicated with a warning if they are
  not strictly increasing, so a malformed source never silently corrupts
  the interpolation.
"""

import json
import lxml.etree as ET
import pandas as pd
import shutil
import os
import logging
import time
from datetime import datetime


# ─────────────────────────────────────────────────────────────────────────────
# LOGGING
# ─────────────────────────────────────────────────────────────────────────────

logging.basicConfig(
    level=logging.WARNING,
    format='%(levelname)s: %(message)s',
    handlers=[
        logging.StreamHandler(),
        logging.FileHandler('calibration_process.log', mode='w')
    ]
)
file_logger = logging.getLogger('file_logger')
file_logger.setLevel(logging.INFO)
file_handler = logging.FileHandler('calibration_process.log', mode='w')
file_handler.setFormatter(logging.Formatter('%(levelname)s: %(message)s'))
file_logger.addHandler(file_handler)


# ─────────────────────────────────────────────────────────────────────────────
# CORE HELPERS — interpolation / extrapolation
# ─────────────────────────────────────────────────────────────────────────────

def _ensure_monotonic(src_x, src_v, label_for_log=""):
    """
    Return (src_x, src_v) sorted by src_x ascending, with duplicate x values
    collapsed (keeping the first occurrence). Logs a warning if the input
    was not already strictly increasing.

    Required because interp_extrap_values assumes strictly increasing src_x;
    a non-monotonic axis would cause the segment search to pick the wrong
    segment silently.
    """
    n = len(src_x)
    if n <= 1:
        return list(src_x), list(src_v)

    strictly_increasing = all(src_x[i] < src_x[i + 1] for i in range(n - 1))
    if strictly_increasing:
        return list(src_x), list(src_v)

    file_logger.warning(
        f"  {label_for_log}: src x-axis not strictly increasing — sorting "
        f"and de-duplicating before interpolation."
    )

    pairs = sorted(zip(src_x, src_v), key=lambda p: p[0])
    out_x, out_v = [], []
    for x, v in pairs:
        if out_x and x == out_x[-1]:
            continue  # drop duplicate x
        out_x.append(x)
        out_v.append(v)
    return out_x, out_v


def interp_extrap_values(src_x, src_v, dest_x, label_for_log=""):
    """
    Linear interpolation inside src_x range, slope-based linear extrapolation
    outside it.

    * Empty src → returns zeros (with warning).
    * Single src point → constant fill.
    * Non-monotonic src_x → auto-sorted/de-duplicated with warning.
    """
    n = len(src_x)
    m = len(dest_x)

    if n == 0:
        file_logger.warning(
            f"  {label_for_log}: empty source — returning zeros."
        )
        return [0.0] * m

    if n == 1:
        return [src_v[0]] * m

    src_x, src_v = _ensure_monotonic(src_x, src_v, label_for_log)
    n = len(src_x)
    if n == 1:
        return [src_v[0]] * m

    slope_left = (
        (src_v[1] - src_v[0]) / (src_x[1] - src_x[0])
        if src_x[1] != src_x[0] else 0.0
    )
    slope_right = (
        (src_v[-1] - src_v[-2]) / (src_x[-1] - src_x[-2])
        if src_x[-1] != src_x[-2] else 0.0
    )

    result = []
    for dx in dest_x:
        if dx <= src_x[0]:
            val = src_v[0] + slope_left * (dx - src_x[0])
        elif dx >= src_x[-1]:
            val = src_v[-1] + slope_right * (dx - src_x[-1])
        else:
            i = 0
            for k in range(n - 1):
                if src_x[k] <= dx < src_x[k + 1]:
                    i = k
                    break
            span = src_x[i + 1] - src_x[i]
            t = (dx - src_x[i]) / span if span != 0 else 0.0
            val = src_v[i] + t * (src_v[i + 1] - src_v[i])
        result.append(val)

    return result


def _parse_floats(elements):
    out = []
    for el in elements:
        if el.text and el.text.strip():
            try:
                out.append(float(el.text.strip()))
            except ValueError:
                pass
    return out


def _parse_strings(elements):
    return [el.text.strip() for el in elements if el.text and el.text.strip()]


def _safe_float_list(lst):
    """Convert a list to floats; silently drop non-numeric items."""
    out = []
    for v in lst:
        try:
            out.append(float(v))
        except (ValueError, TypeError):
            pass
    return out


def _try_float_list(strings):
    """
    Try to convert every string to float; return (floats, True) on full
    success, or ([], False) if any fail. Useful for "are these axis values
    numeric?" checks where partial conversion would be misleading.
    """
    out = []
    for v in strings:
        try:
            out.append(float(v))
        except (ValueError, TypeError):
            return [], False
    return out, True


def _axis_values(axis_entry):
    """
    Extract the value list from one axis entry in a JSON calibration file.
    Handles both list-style and dict-style ({"values": [...]} ) axes.
    """
    if isinstance(axis_entry, list):
        return axis_entry
    if isinstance(axis_entry, dict):
        return axis_entry.get('values', [])
    return []


def _nearest_neighbour_index_map(src_n, dest_n):
    """Return a length-dest_n list of src indices via NN sampling."""
    if src_n == 0 or dest_n == 0:
        return []
    if src_n == 1:
        return [0] * dest_n
    out = []
    for i in range(dest_n):
        src_idx = round(i * (src_n - 1) / max(dest_n - 1, 1))
        out.append(max(0, min(src_n - 1, src_idx)))
    return out


def _nearest_neighbour_label_map(src_labels, dest_labels):
    """
    For each dest label, find the src index whose label matches exactly.
    Falls back to nearest-position index for unmatched dest labels.
    Used for enum/string axes where indices have no relationship to labels.
    """
    src_n = len(src_labels)
    dest_n = len(dest_labels)
    if src_n == 0 or dest_n == 0:
        return []
    # Build src label → first index lookup
    src_index = {}
    for i, s in enumerate(src_labels):
        if s not in src_index:
            src_index[s] = i

    nn_fallback = _nearest_neighbour_index_map(src_n, dest_n)
    out = []
    for i, d in enumerate(dest_labels):
        if d in src_index:
            out.append(src_index[d])
        else:
            out.append(nn_fallback[i])
    return out


def _format_num(v):
    """Format a float for XML writeback. Centralised so format changes
    only need to happen in one place."""
    return f"{v}"


# ─────────────────────────────────────────────────────────────────────────────
# FILE TYPE DETECTION
# ─────────────────────────────────────────────────────────────────────────────

def _file_type(path):
    return 'json' if os.path.splitext(path)[1].lower() == '.json' else 'cdfx'


# ─────────────────────────────────────────────────────────────────────────────
# XML helpers — destination is always CDFX
# ─────────────────────────────────────────────────────────────────────────────

def filter_destination_file(original_dest, filtered_dest, labels_to_keep, namespace):
    tree = ET.parse(original_dest)
    root = tree.getroot()
    for sw in root.findall('.//SW-INSTANCE', namespaces=namespace):
        sn = sw.find('SHORT-NAME', namespaces=namespace)
        if sn is None or sn.text.strip() not in labels_to_keep:
            parent = sw.getparent()
            if parent is not None:
                parent.remove(sw)
    tree.write(filtered_dest, encoding='utf-8', xml_declaration=True)


def merge_updated_instances(main_dest, updated_dest, labels_to_merge, namespace):
    main_tree = ET.parse(main_dest)
    main_root = main_tree.getroot()
    updated_tree = ET.parse(updated_dest)
    updated_root = updated_tree.getroot()

    main_sw_map = {}
    for sw in main_root.findall('.//SW-INSTANCE', namespaces=namespace):
        sn = sw.find('SHORT-NAME', namespaces=namespace)
        if sn is not None and sn.text.strip() in labels_to_merge:
            main_sw_map[sn.text.strip()] = sw

    for sw in updated_root.findall('.//SW-INSTANCE', namespaces=namespace):
        sn = sw.find('SHORT-NAME', namespaces=namespace)
        if sn is not None and sn.text.strip() in labels_to_merge:
            old_sw = main_sw_map.get(sn.text.strip())
            if old_sw is not None:
                parent = old_sw.getparent()
                if parent is not None:
                    parent.replace(old_sw, sw)
    main_tree.write(main_dest, encoding='utf-8', xml_declaration=True)


def extract_labels_with_c(cdfx_file, namespace):
    """
    Lightweight summary extraction used for inventorying labels in source
    and destination files. Not used by the per-type update handlers —
    those work directly against lxml elements.
    """
    try:
        tree = ET.parse(cdfx_file)
        root = tree.getroot()
    except ET.XMLSyntaxError as e:
        logging.error(f"XML Syntax Error while parsing '{cdfx_file}': {e}")
        return pd.DataFrame()
    except FileNotFoundError:
        logging.error(f"File not found: '{cdfx_file}'")
        return pd.DataFrame()
    except Exception as e:
        logging.error(f"Unexpected error while parsing '{cdfx_file}': {e}")
        return pd.DataFrame()

    data = []
    for sw in root.findall(".//SW-INSTANCE", namespaces=namespace):
        sn = sw.find("SHORT-NAME", namespaces=namespace)
        if sn is None or not sn.text:
            continue
        label = sn.text.strip()

        if label.endswith(('_C', '_CW', '_c')):
            v = sw.find("SW-VALUE-CONT/SW-VALUES-PHYS/V", namespaces=namespace)
            if v is not None and v.text and v.text.strip():
                data.append({'label': label, 'values': v.text.strip()})
            else:
                vt = sw.find("SW-VALUE-CONT/SW-VALUES-PHYS/VT", namespaces=namespace)
                if vt is not None and vt.text and vt.text.strip():
                    data.append({'label': label, 'values': vt.text.strip()})

        elif label.endswith('_CA'):
            dim = sw.find("SW-VALUE-CONT/SW-ARRAYSIZE/V", namespaces=namespace)
            if dim is not None and dim.text and dim.text.strip():
                vt_elements = sw.findall(".//SW-VALUE-CONT/SW-VALUES-PHYS/VT",
                                         namespaces=namespace)
                value = [vt.text.strip() for vt in vt_elements
                         if vt.text and vt.text.strip()]
                if not value:
                    v_elements = sw.findall(".//SW-VALUE-CONT/SW-VALUES-PHYS/V",
                                            namespaces=namespace)
                    value = [v.text.strip() for v in v_elements
                             if v.text and v.text.strip()]
                data.append({'label': label, 'dimen': dim.text, 'values': value})

        elif label.endswith(('_MAP', '_M')):
            sw_values = sw.find("SW-VALUE-CONT/SW-VALUES-PHYS", namespaces=namespace)
            if sw_values is not None:
                value_dict = {}
                for vg in sw_values.findall("VG", namespaces=namespace):
                    label_elem = vg.find("LABEL", namespaces=namespace)
                    if label_elem is not None and label_elem.text and label_elem.text.strip():
                        key = label_elem.text.strip()
                        try:
                            key = float(key)
                        except ValueError:
                            pass
                        values = _parse_floats(vg.findall("V", namespaces=namespace))
                        value_dict[key] = values
                axis = sw.findall(".//SW-AXIS-CONT", namespaces=namespace)
                x_vals, y_vals = [], []
                if axis:
                    sw_vp = axis[0].find("SW-VALUES-PHYS", namespaces=namespace)
                    if sw_vp is not None:
                        x_vals = _parse_floats(sw_vp.findall("V", namespaces=namespace))
                    try:
                        y_vals = sorted(value_dict.keys(),
                                        key=lambda x: (float(x) if isinstance(x, float) else x))
                    except TypeError:
                        y_vals = list(value_dict.keys())
                data.append({'label': label, 'values': value_dict,
                             'x_dim': len(x_vals), 'y_dim': len(y_vals),
                             'x_dim_val': x_vals, 'y_dim_val': y_vals})

        elif label.endswith(('_T', '_CUR', '_Cur')):
            axis = sw.findall(".//SW-VALUES-PHYS", namespaces=namespace)
            if axis:
                z_vals = _parse_strings(axis[0].findall("V", namespaces=namespace))
                if not z_vals:
                    z_vals = _parse_strings(axis[0].findall("VT", namespaces=namespace))
                x_vals = []
                if len(axis) > 1:
                    x_vals = _parse_strings(axis[1].findall("V", namespaces=namespace))
                    if not x_vals:
                        x_vals = _parse_strings(axis[1].findall("VT", namespaces=namespace))
                data.append({'label': label, 'x_values': x_vals,
                             'z_values': z_vals, 'x_dim': len(x_vals)})

    return pd.DataFrame(data)


def build_label_index(root, namespace):
    label_index = {}
    for sw in root.findall(".//SW-INSTANCE", namespaces=namespace):
        sn = sw.find("SHORT-NAME", namespaces=namespace)
        if sn is not None and sn.text and sn.text.strip():
            label_index[sn.text.strip()] = sw
    return label_index


# ─────────────────────────────────────────────────────────────────────────────
# AXIS UTILITIES — used by _MAP handlers
# ─────────────────────────────────────────────────────────────────────────────

def _read_axis_strings_and_numbers(svp_elem, namespace):
    """
    Read an SW-VALUES-PHYS axis block, preferring VT (string) elements but
    falling back to V (numeric). Returns (string_list, numeric_list_or_None,
    is_numeric, element_list).

    is_numeric is True only when ALL values parse cleanly as floats AND
    the values came from <V> elements (not <VT>, which are explicitly
    string-typed even if their text happens to be numeric).
    """
    if svp_elem is None:
        return [], None, False, []
    # Try V first (numeric convention), then VT (string convention)
    v_elems = svp_elem.findall("V", namespaces=namespace)
    vt_elems = svp_elem.findall("VT", namespaces=namespace)
    if v_elems:
        elems = v_elems
        strings = [e.text.strip() if e.text else "" for e in elems]
        nums, ok = _try_float_list(strings)
        return strings, (nums if ok else None), ok, elems
    elif vt_elems:
        elems = vt_elems
        strings = [e.text.strip() if e.text else "" for e in elems]
        # VT axes are enum by definition
        return strings, None, False, elems
    return [], None, False, []


def _resample_axis(src_axis_vals, dest_elems, label, axis_name):
    """
    Resample a single numeric axis (x or y of a _MAP) from src onto dest.

    The dest axis count is preserved. Source positions are normalised onto
    the dest index range so the two coordinate spaces match. Updates the
    text of dest_elems in place and returns the list of new float values.

    Note: this is a code-internal helper for the case where the dest axis
    has NO physical meaning we want to preserve (i.e., we're aligning
    indices). For physical-coordinate bilinear data resampling, the src
    and dest axes are used as physical coordinates directly via
    interp_extrap_values — that's done in _update_map_2d, not here.
    """
    dest_n = len(dest_elems)
    src_n = len(src_axis_vals)
    if dest_n == 0:
        return []
    if src_n == 0:
        return _parse_floats(dest_elems)
    if src_n == 1:
        for elem in dest_elems:
            elem.text = _format_num(src_axis_vals[0])
        return [src_axis_vals[0]] * dest_n
    src_x_norm = [i * (dest_n - 1) / (src_n - 1) for i in range(src_n)]
    dest_pos = list(range(dest_n))
    new_vals = interp_extrap_values(
        src_x_norm, src_axis_vals, dest_pos, f"{label} ({axis_name})")
    for i, elem in enumerate(dest_elems):
        elem.text = _format_num(new_vals[i])
    return new_vals


# ─────────────────────────────────────────────────────────────────────────────
# PER-TYPE UPDATE HANDLERS — CDFX source
# ─────────────────────────────────────────────────────────────────────────────

def _update_scalar_cdfx(label, correct_sw, incorrect_sw, row, namespace):
    """Scalar from CDFX source. Returns (status, record_or_None)."""
    correct_value = row['values_correct']
    incorrect_value = row.get('values_incorrect', None)

    sw_unit_incorrect = incorrect_sw.find(
        ".//SW-VALUE-CONT/UNIT-DISPLAY-NAME", namespaces=namespace)
    sw_unit_correct = correct_sw.find(
        ".//SW-VALUE-CONT/UNIT-DISPLAY-NAME", namespaces=namespace)
    if sw_unit_incorrect is not None and sw_unit_correct is not None:
        sw_unit_incorrect.text = sw_unit_correct.text

    if correct_value == incorrect_value:
        return 'no_change', None

    sw_val_incorrect = incorrect_sw.find(
        ".//SW-VALUE-CONT/SW-VALUES-PHYS/V", namespaces=namespace)
    sw_val_correct = correct_sw.find(
        ".//SW-VALUE-CONT/SW-VALUES-PHYS/V", namespaces=namespace)
    if sw_val_incorrect is None:
        sw_val_incorrect = incorrect_sw.find(
            ".//SW-VALUE-CONT/SW-VALUES-PHYS/VT", namespaces=namespace)
    if sw_val_correct is None:
        sw_val_correct = correct_sw.find(
            ".//SW-VALUE-CONT/SW-VALUES-PHYS/VT", namespaces=namespace)
    if sw_val_incorrect is None or sw_val_correct is None:
        return 'skip', None

    sw_val_incorrect.text = f"{correct_value}"
    return 'updated', {'label': label, 'values': correct_value, 'status': 'Updated'}


def _update_array_1d_cdfx(label, correct_sw, incorrect_sw, namespace):
    """
    _CA from CDFX source. Index-based resampling since arrays have no axis.
    Returns 'no_change' if dest values are already exactly what we'd write.
    """
    correct_values_phys = correct_sw.find(
        "SW-VALUE-CONT/SW-VALUES-PHYS", namespaces=namespace)
    incorrect_values_phys = incorrect_sw.find(
        "SW-VALUE-CONT/SW-VALUES-PHYS", namespaces=namespace)
    if correct_values_phys is None or incorrect_values_phys is None:
        file_logger.warning(f"  {label}: Missing SW-VALUES-PHYS — skipping")
        return 'skip', None

    src_str = _parse_strings(
        correct_values_phys.findall("VT", namespaces=namespace))
    if not src_str:
        src_str = _parse_strings(
            correct_values_phys.findall("V", namespaces=namespace))

    dest_elems = incorrect_values_phys.findall("VT", namespaces=namespace)
    if not dest_elems:
        dest_elems = incorrect_values_phys.findall("V", namespaces=namespace)

    src_n = len(src_str)
    dest_n = len(dest_elems)
    if src_n == 0 or dest_n == 0:
        return 'skip', None

    old_texts = [e.text or "" for e in dest_elems]

    src_v_float, numeric = _try_float_list(src_str)

    if numeric:
        src_x = ([i * (dest_n - 1) / (src_n - 1) for i in range(src_n)]
                 if src_n > 1 else [0.0])
        new_vals = interp_extrap_values(src_x, src_v_float, list(range(dest_n)),
                                        label)
        new_texts = [_format_num(v) for v in new_vals]
    else:
        idx_map = _nearest_neighbour_index_map(src_n, dest_n)
        new_texts = [src_str[i] for i in idx_map]

    if [t.strip() for t in old_texts] == [t.strip() for t in new_texts]:
        return 'no_change', None

    for elem, text in zip(dest_elems, new_texts):
        elem.text = text

    return 'updated', {'label': label, 'dimen': dest_n,
                       'values': new_texts, 'status': 'Updated'}


def _update_map_2d_cdfx(label, correct_sw, incorrect_sw, namespace):
    """
    _MAP from CDFX source.

    Decision tree:
      1. If both axes are numeric → physical-coordinate bilinear interp/extrap.
      2. If either axis is non-numeric (enum) → label-matched lookup on the
         enum dimension; numeric interp on the other (if numeric).
      3. If neither axis is usable → fall back to index-based bilinear.
    """
    sw_val_correct = correct_sw.find(
        "SW-VALUE-CONT/SW-VALUES-PHYS", namespaces=namespace)
    sw_val_incorrect = incorrect_sw.find(
        "SW-VALUE-CONT/SW-VALUES-PHYS", namespaces=namespace)
    if sw_val_correct is None or sw_val_incorrect is None:
        return 'skip', None

    axis_correct = correct_sw.findall(".//SW-AXIS-CONT", namespaces=namespace)
    axis_incorrect = incorrect_sw.findall(".//SW-AXIS-CONT", namespaces=namespace)

    # ── X-axis (axis[0]) ────────────────────────────────────────────────────
    src_x_strs, src_x_nums, src_x_is_num = [], None, False
    dest_x_strs, dest_x_nums, dest_x_is_num = [], None, False
    dest_x_elems = []
    if axis_correct and axis_incorrect:
        svp_c = axis_correct[0].find("SW-VALUES-PHYS", namespaces=namespace)
        svp_d = axis_incorrect[0].find("SW-VALUES-PHYS", namespaces=namespace)
        src_x_strs, src_x_nums, src_x_is_num, _ = \
            _read_axis_strings_and_numbers(svp_c, namespace)
        dest_x_strs, dest_x_nums, dest_x_is_num, dest_x_elems = \
            _read_axis_strings_and_numbers(svp_d, namespace)

    # ── Y-axis (axis[1]) ────────────────────────────────────────────────────
    src_y_strs, src_y_nums, src_y_is_num = [], None, False
    dest_y_strs, dest_y_nums, dest_y_is_num = [], None, False
    dest_y_elems = []
    if len(axis_correct) > 1 and len(axis_incorrect) > 1:
        svp_c1 = axis_correct[1].find("SW-VALUES-PHYS", namespaces=namespace)
        svp_d1 = axis_incorrect[1].find("SW-VALUES-PHYS", namespaces=namespace)
        src_y_strs, src_y_nums, src_y_is_num, _ = \
            _read_axis_strings_and_numbers(svp_c1, namespace)
        dest_y_strs, dest_y_nums, dest_y_is_num, dest_y_elems = \
            _read_axis_strings_and_numbers(svp_d1, namespace)

    x_axis_numeric = src_x_is_num and dest_x_is_num
    y_axis_numeric = src_y_is_num and dest_y_is_num

    # ── Read data cells from source ────────────────────────────────────────
    correct_vgs = sw_val_correct.findall("VG", namespaces=namespace)
    incorrect_vgs = sw_val_incorrect.findall("VG", namespaces=namespace)
    src_row_count = len(correct_vgs)
    dest_row_count = len(incorrect_vgs)
    if src_row_count == 0 or dest_row_count == 0:
        return 'skip', None

    src_row_labels = []
    src_row_data = []
    for vg in correct_vgs:
        lbl_el = vg.find("LABEL", namespaces=namespace)
        lbl_text = lbl_el.text.strip() if (lbl_el is not None and lbl_el.text) else ""
        src_row_labels.append(lbl_text)
        src_row_data.append(_parse_floats(vg.findall("V", namespaces=namespace)))

    dest_row_labels = []
    for vg in incorrect_vgs:
        lbl_el = vg.find("LABEL", namespaces=namespace)
        lbl_text = lbl_el.text.strip() if (lbl_el is not None and lbl_el.text) else ""
        dest_row_labels.append(lbl_text)

    row_lengths = [len(r) for r in src_row_data]
    max_src_cols = max(row_lengths, default=0)
    if max_src_cols == 0:
        return 'skip', None
    if any(rl != max_src_cols for rl in row_lengths):
        file_logger.warning(
            f"  {label}: source rows are ragged (lengths={row_lengths}) — "
            f"padding short rows with their last value."
        )

    # ── Update X-axis text ─────────────────────────────────────────────────
    # For enum axes: leave dest axis labels alone if they already represent
    # the same set as src — the data resampling does label-matched lookup,
    # so dest's own enum ordering must be preserved.
    x_vals_new_display = []
    if x_axis_numeric and len(src_x_nums) >= 2 and dest_x_nums:
        x_vals_new_display = _resample_axis(
            src_x_nums, dest_x_elems, label, "x-axis")
    elif x_axis_numeric and len(src_x_nums) == 1 and dest_x_elems:
        for elem in dest_x_elems:
            elem.text = _format_num(src_x_nums[0])
        x_vals_new_display = [src_x_nums[0]] * len(dest_x_elems)
    else:
        # Enum or unusable → leave dest axis untouched
        x_vals_new_display = dest_x_strs or _parse_floats(dest_x_elems)

    # ── Update Y-axis text ─────────────────────────────────────────────────
    y_vals_new_display = []
    if y_axis_numeric and len(src_y_nums) >= 2 and dest_y_nums:
        y_vals_new_display = _resample_axis(
            src_y_nums, dest_y_elems, label, "y-axis")
    elif y_axis_numeric and len(src_y_nums) == 1 and dest_y_elems:
        for elem in dest_y_elems:
            elem.text = _format_num(src_y_nums[0])
        y_vals_new_display = [src_y_nums[0]] * len(dest_y_elems)
    else:
        # Enum or unusable → leave dest axis untouched
        y_vals_new_display = dest_y_strs or _parse_floats(dest_y_elems)

    # ── Resample data cells ────────────────────────────────────────────────
    # Decide row mapping strategy
    # Note: row labels in VG come from the y-axis (each VG = one y position).
    if y_axis_numeric and src_y_nums and dest_y_nums \
            and len(src_y_nums) == src_row_count \
            and len(dest_y_nums) == dest_row_count:
        # Sort src rows by their y-coordinate so the axis is monotonic
        # before physical-coordinate interpolation.
        pairs = sorted(zip(src_y_nums, src_row_data), key=lambda p: p[0])
        src_y_sorted = [p[0] for p in pairs]
        src_rows_sorted = [p[1] for p in pairs]
        row_strategy = ('numeric', src_y_sorted, dest_y_nums, src_rows_sorted)
    elif src_y_strs and dest_row_labels \
            and len(src_y_strs) == src_row_count:
        # Enum row axis → label-matched lookup
        # Prefer matching dest VG LABELs (which mirror the y-axis values)
        idx_map = _nearest_neighbour_label_map(src_y_strs, dest_row_labels)
        row_strategy = ('enum_label', idx_map, src_row_data)
    else:
        # No usable y-axis → index normalisation
        src_row_norm = ([i * (dest_row_count - 1) / (src_row_count - 1)
                         for i in range(src_row_count)]
                        if src_row_count >= 2 else [0.0])
        row_strategy = ('index', src_row_norm, src_row_data)

    # Decide column mapping strategy (same logic, applied to x-axis)
    if x_axis_numeric and src_x_nums and dest_x_nums \
            and len(src_x_nums) == max_src_cols:
        src_x_sorted_pairs = sorted(enumerate(src_x_nums), key=lambda p: p[1])
        src_x_sorted = [p[1] for p in src_x_sorted_pairs]
        col_perm = [p[0] for p in src_x_sorted_pairs]
        col_strategy = ('numeric', src_x_sorted, dest_x_nums, col_perm)
    elif src_x_strs and len(src_x_strs) == max_src_cols:
        col_strategy = ('enum_label', src_x_strs)
    else:
        src_col_norm = ([j * (1) for j in range(max_src_cols)]
                        if max_src_cols >= 2 else [0.0])
        col_strategy = ('index', src_col_norm)

    # Iterate dest rows
    for row_i, dest_vg in enumerate(incorrect_vgs):
        dest_v_elems = dest_vg.findall("V", namespaces=namespace)
        dest_col_count = len(dest_v_elems)
        if dest_col_count == 0:
            continue

        # Step 1: produce an intermediate row vector at this dest row position
        # The intermediate vector has one entry per src column (max_src_cols).
        intermediate = []
        if row_strategy[0] == 'numeric':
            _, src_y_sorted, dest_y_nums_local, src_rows_sorted = row_strategy
            dest_y_pos = dest_y_nums_local[row_i]
            for col_j in range(max_src_cols):
                col_signal = [
                    r[col_j] if col_j < len(r) else (r[-1] if r else 0.0)
                    for r in src_rows_sorted
                ]
                v = interp_extrap_values(
                    src_y_sorted, col_signal, [dest_y_pos],
                    f"{label} (y-resample col={col_j})")[0]
                intermediate.append(v)
        elif row_strategy[0] == 'enum_label':
            _, idx_map, src_rows = row_strategy
            chosen_row = src_rows[idx_map[row_i]] if idx_map else []
            for col_j in range(max_src_cols):
                intermediate.append(
                    chosen_row[col_j] if col_j < len(chosen_row)
                    else (chosen_row[-1] if chosen_row else 0.0)
                )
        else:  # 'index'
            _, src_row_norm, src_rows = row_strategy
            for col_j in range(max_src_cols):
                col_signal = [
                    r[col_j] if col_j < len(r) else (r[-1] if r else 0.0)
                    for r in src_rows
                ]
                v = interp_extrap_values(
                    src_row_norm, col_signal, [row_i],
                    f"{label} (row-index col={col_j})")[0]
                intermediate.append(v)

        # Step 2: project intermediate vector onto dest columns
        if col_strategy[0] == 'numeric':
            _, src_x_sorted, dest_x_nums_local, col_perm = col_strategy
            # Reorder intermediate by col_perm so it aligns with sorted src_x
            intermediate_sorted = [intermediate[j] for j in col_perm]
            final_col_vals = interp_extrap_values(
                src_x_sorted, intermediate_sorted, dest_x_nums_local,
                f"{label} (x-resample row={row_i})")
        elif col_strategy[0] == 'enum_label':
            _, src_x_labels = col_strategy
            idx_map = _nearest_neighbour_label_map(src_x_labels, dest_x_strs)
            # idx_map may be empty if dest_x_strs is empty (no axis info)
            if idx_map:
                final_col_vals = [intermediate[i] for i in idx_map]
            else:
                # No dest x-axis info → length-based NN
                idx_map = _nearest_neighbour_index_map(max_src_cols, dest_col_count)
                final_col_vals = [intermediate[i] for i in idx_map]
        else:  # 'index'
            if max_src_cols >= 2:
                src_col_norm_local = [
                    j * (dest_col_count - 1) / (max_src_cols - 1)
                    for j in range(max_src_cols)
                ]
                final_col_vals = interp_extrap_values(
                    src_col_norm_local, intermediate, list(range(dest_col_count)),
                    f"{label} (col-index row={row_i})")
            else:
                final_col_vals = [intermediate[0]] * dest_col_count

        for col_i, v_elem in enumerate(dest_v_elems):
            v_elem.text = _format_num(final_col_vals[col_i])

        # Update VG LABEL only when y-axis was numerically resampled.
        # For enum y-axes we preserve dest VG labels (and the data was
        # already label-matched into the correct rows).
        if y_axis_numeric:
            lbl_el = dest_vg.find("LABEL", namespaces=namespace)
            if lbl_el is not None and y_vals_new_display \
                    and row_i < len(y_vals_new_display):
                new_lbl = y_vals_new_display[row_i]
                lbl_el.text = (_format_num(new_lbl) if isinstance(new_lbl, (int, float))
                               else str(new_lbl))

    value_dict = {}
    for vg in incorrect_vgs:
        lbl_el = vg.find("LABEL", namespaces=namespace)
        key = lbl_el.text.strip() if (lbl_el is not None and lbl_el.text) \
            else str(id(vg))
        value_dict[key] = _parse_floats(vg.findall("V", namespaces=namespace))

    return 'updated', {'label': label, 'values': value_dict,
                       'x_dim': len(x_vals_new_display),
                       'y_dim': len(y_vals_new_display),
                       'x_dim_val': x_vals_new_display,
                       'y_dim_val': y_vals_new_display,
                       'status': 'Updated'}


def _update_curve_1d_cdfx(label, correct_sw, incorrect_sw, namespace):
    """
    _T / _CUR / _Cur from CDFX source.

    Branches:
      1. numeric z + numeric x + lengths match → physical-coordinate
         interp/extrap. Dest x-axis left untouched so (x, z) pairs stay
         self-consistent.
      2. numeric z, x-axis unusable           → index-based fallback.
      3. non-numeric z                         → nearest-neighbour by index.
    """
    axis_correct = correct_sw.findall(".//SW-VALUES-PHYS", namespaces=namespace)
    axis_incorrect = incorrect_sw.findall(".//SW-VALUES-PHYS", namespaces=namespace)
    if not axis_correct or not axis_incorrect:
        return 'skip', None

    src_z_str = _parse_strings(axis_correct[0].findall("V", namespaces=namespace))
    if not src_z_str:
        src_z_str = _parse_strings(axis_correct[0].findall("VT", namespaces=namespace))

    dest_z_elems = axis_incorrect[0].findall("V", namespaces=namespace)
    if not dest_z_elems:
        dest_z_elems = axis_incorrect[0].findall("VT", namespaces=namespace)

    src_z_n = len(src_z_str)
    dest_z_n = len(dest_z_elems)
    if src_z_n == 0 or dest_z_n == 0:
        return 'skip', None

    src_x_str = []
    dest_x_elems = []
    if len(axis_correct) > 1 and len(axis_incorrect) > 1:
        src_x_str = _parse_strings(axis_correct[1].findall("V", namespaces=namespace))
        if not src_x_str:
            src_x_str = _parse_strings(axis_correct[1].findall("VT", namespaces=namespace))
        dest_x_elems = axis_incorrect[1].findall("V", namespaces=namespace)
        if not dest_x_elems:
            dest_x_elems = axis_incorrect[1].findall("VT", namespaces=namespace)

    src_z_float, numeric_z = _try_float_list(src_z_str)
    src_x_float, src_x_ok = _try_float_list(src_x_str) if src_x_str else ([], False)
    dest_x_float = _parse_floats(dest_x_elems) if dest_x_elems else []
    numeric_x = src_x_ok and bool(dest_x_float)

    old_z_texts = [e.text or "" for e in dest_z_elems]

    if numeric_z and numeric_x and len(src_x_float) == src_z_n:
        # Real physical-coordinate resample at dest's existing x-grid.
        # Dest x-axis is left untouched — (x, z) pairs stay self-consistent.
        new_z = interp_extrap_values(
            src_x_float, src_z_float, dest_x_float, f"{label} (z@dest_x)")
        new_z_texts = [_format_num(v) for v in new_z]
    elif numeric_z and numeric_x and len(src_x_float) != src_z_n:
        file_logger.warning(
            f"  {label}: src z length ({src_z_n}) != src x length "
            f"({len(src_x_float)}) — falling back to index-based interpolation."
        )
        src_x_norm = ([i * (dest_z_n - 1) / (src_z_n - 1) for i in range(src_z_n)]
                      if src_z_n >= 2 else [0.0])
        new_z = interp_extrap_values(
            src_x_norm, src_z_float, list(range(dest_z_n)), f"{label} (index fallback)")
        new_z_texts = [_format_num(v) for v in new_z]
    elif numeric_z:
        src_x_norm = ([i * (dest_z_n - 1) / (src_z_n - 1) for i in range(src_z_n)]
                      if src_z_n >= 2 else [0.0])
        new_z = interp_extrap_values(
            src_x_norm, src_z_float, list(range(dest_z_n)), f"{label} (index)")
        new_z_texts = [_format_num(v) for v in new_z]
    else:
        idx_map = _nearest_neighbour_index_map(src_z_n, dest_z_n)
        new_z_texts = [src_z_str[i] for i in idx_map]

    if [t.strip() for t in old_z_texts] == [t.strip() for t in new_z_texts]:
        return 'no_change', None

    for elem, text in zip(dest_z_elems, new_z_texts):
        elem.text = text

    x_vals = [e.text for e in dest_x_elems] if dest_x_elems else []
    z_vals = [e.text for e in dest_z_elems]
    return 'updated', {'label': label, 'x_values': x_vals,
                       'z_values': z_vals, 'x_dim': len(x_vals),
                       'status': 'Updated'}


# ─────────────────────────────────────────────────────────────────────────────
# PER-TYPE UPDATE HANDLERS — JSON source
# ─────────────────────────────────────────────────────────────────────────────

def _update_scalar_json(label, src_entry, dest_sw, namespace):
    src_val = src_entry.get('value')
    if src_val is None:
        return 'skip', None
    if isinstance(src_val, (list, dict)):
        file_logger.warning(
            f"  {label}: scalar expected but value is "
            f"{type(src_val).__name__} — skipping")
        return 'skip', None

    if 'unit' in src_entry:
        unit_elem = dest_sw.find(".//SW-VALUE-CONT/UNIT-DISPLAY-NAME",
                                 namespaces=namespace)
        if unit_elem is not None:
            unit_elem.text = str(src_entry['unit'])

    dest_elem = dest_sw.find(".//SW-VALUE-CONT/SW-VALUES-PHYS/V",
                             namespaces=namespace)
    if dest_elem is None:
        dest_elem = dest_sw.find(".//SW-VALUE-CONT/SW-VALUES-PHYS/VT",
                                 namespaces=namespace)
    if dest_elem is None:
        file_logger.warning(f"  {label}: no V/VT element in dest — skipping")
        return 'skip', None

    new_text = str(src_val)
    if dest_elem.text and dest_elem.text.strip() == new_text:
        return 'no_change', None
    dest_elem.text = new_text
    return 'updated', {'label': label, 'values': src_val, 'status': 'Updated'}


def _update_array_1d_json(label, src_entry, dest_sw, namespace):
    src_vals = src_entry.get('value', [])
    if not isinstance(src_vals, list):
        file_logger.warning(
            f"  {label}: _CA expected list but value is "
            f"{type(src_vals).__name__} — skipping")
        return 'skip', None
    if not src_vals:
        return 'skip', None

    dest_phys = dest_sw.find("SW-VALUE-CONT/SW-VALUES-PHYS", namespaces=namespace)
    if dest_phys is None:
        file_logger.warning(f"  {label}: no SW-VALUES-PHYS in dest — skipping")
        return 'skip', None

    dest_elems = dest_phys.findall("VT", namespaces=namespace)
    if not dest_elems:
        dest_elems = dest_phys.findall("V", namespaces=namespace)

    src_n = len(src_vals)
    dest_n = len(dest_elems)
    if src_n == 0 or dest_n == 0:
        return 'skip', None

    old_texts = [e.text or "" for e in dest_elems]

    try:
        src_float = [float(v) for v in src_vals]
        numeric = True
    except (ValueError, TypeError):
        numeric = False

    if numeric:
        src_x = ([i * (dest_n - 1) / (src_n - 1) for i in range(src_n)]
                 if src_n > 1 else [0.0])
        new_vals = interp_extrap_values(src_x, src_float, list(range(dest_n)), label)
        new_texts = [_format_num(v) for v in new_vals]
    else:
        idx_map = _nearest_neighbour_index_map(src_n, dest_n)
        new_texts = [str(src_vals[i]) for i in idx_map]

    if [t.strip() for t in old_texts] == [t.strip() for t in new_texts]:
        return 'no_change', None

    for elem, text in zip(dest_elems, new_texts):
        elem.text = text

    return 'updated', {'label': label, 'dimen': dest_n,
                       'values': new_texts, 'status': 'Updated'}


def _update_map_2d_json(label, src_entry, dest_sw, namespace):
    """
    _MAP from JSON source. Uses the same physical-coordinate / label-match
    decision tree as the CDFX path.
    """
    src_value = src_entry.get('value', [])
    src_axes = src_entry.get('axes', [])
    if not isinstance(src_value, list) or not src_value:
        file_logger.warning(f"  {label}: _MAP value missing or not a list — skipping")
        return 'skip', None
    if not all(isinstance(r, list) for r in src_value):
        file_logger.warning(f"  {label}: _MAP rows are not all lists — skipping")
        return 'skip', None
    if not isinstance(src_axes, list):
        src_axes = []

    dest_phys = dest_sw.find("SW-VALUE-CONT/SW-VALUES-PHYS", namespaces=namespace)
    if dest_phys is None:
        return 'skip', None

    axis_dest = dest_sw.findall(".//SW-AXIS-CONT", namespaces=namespace)

    # Source axis values (from JSON)
    src_x_raw = _axis_values(src_axes[0]) if src_axes else []
    src_y_raw = _axis_values(src_axes[1]) if len(src_axes) > 1 else []
    src_x_strs = [str(v) for v in src_x_raw]
    src_y_strs = [str(v) for v in src_y_raw]
    src_x_nums, src_x_is_num = _try_float_list(src_x_strs)
    src_y_nums, src_y_is_num = _try_float_list(src_y_strs)

    # Destination axis values (from CDFX)
    dest_x_strs, dest_x_nums, dest_x_is_num, dest_x_elems = [], None, False, []
    if axis_dest:
        svp_d = axis_dest[0].find("SW-VALUES-PHYS", namespaces=namespace)
        dest_x_strs, dest_x_nums, dest_x_is_num, dest_x_elems = \
            _read_axis_strings_and_numbers(svp_d, namespace)

    dest_y_strs, dest_y_nums, dest_y_is_num, dest_y_elems = [], None, False, []
    if len(axis_dest) > 1:
        svp_d1 = axis_dest[1].find("SW-VALUES-PHYS", namespaces=namespace)
        dest_y_strs, dest_y_nums, dest_y_is_num, dest_y_elems = \
            _read_axis_strings_and_numbers(svp_d1, namespace)

    x_axis_numeric = src_x_is_num and dest_x_is_num
    y_axis_numeric = src_y_is_num and dest_y_is_num

    # Read dest VG row labels
    dest_vgs = dest_phys.findall("VG", namespaces=namespace)
    dest_row_count = len(dest_vgs)
    dest_row_labels = []
    for vg in dest_vgs:
        lbl_el = vg.find("LABEL", namespaces=namespace)
        dest_row_labels.append(
            lbl_el.text.strip() if (lbl_el is not None and lbl_el.text) else ""
        )

    src_row_count = len(src_value)
    if src_row_count == 0 or dest_row_count == 0:
        return 'skip', None

    # Pre-convert src cells to floats (silently zero on failure)
    src_row_data = []
    for r in src_value:
        floats = []
        for v in r:
            try:
                floats.append(float(v))
            except (ValueError, TypeError):
                floats.append(0.0)
        src_row_data.append(floats)

    row_lengths = [len(r) for r in src_row_data]
    max_src_cols = max(row_lengths, default=0)
    if max_src_cols == 0:
        return 'skip', None
    if any(rl != max_src_cols for rl in row_lengths):
        file_logger.warning(
            f"  {label}: source rows are ragged (lengths={row_lengths}) — "
            f"padding short rows with their last value."
        )

    # Write dest axes — preserve dest enum ordering when axes are non-numeric
    x_vals_new_display = []
    if x_axis_numeric and len(src_x_nums) >= 2 and dest_x_nums:
        x_vals_new_display = _resample_axis(src_x_nums, dest_x_elems, label, "x-axis")
    elif x_axis_numeric and len(src_x_nums) == 1 and dest_x_elems:
        for elem in dest_x_elems:
            elem.text = _format_num(src_x_nums[0])
        x_vals_new_display = [src_x_nums[0]] * len(dest_x_elems)
    else:
        x_vals_new_display = dest_x_strs or _parse_floats(dest_x_elems)

    y_vals_new_display = []
    if y_axis_numeric and len(src_y_nums) >= 2 and dest_y_nums:
        y_vals_new_display = _resample_axis(src_y_nums, dest_y_elems, label, "y-axis")
    elif y_axis_numeric and len(src_y_nums) == 1 and dest_y_elems:
        for elem in dest_y_elems:
            elem.text = _format_num(src_y_nums[0])
        y_vals_new_display = [src_y_nums[0]] * len(dest_y_elems)
    else:
        y_vals_new_display = dest_y_strs or _parse_floats(dest_y_elems)

    # Build row strategy
    if y_axis_numeric and src_y_nums and dest_y_nums \
            and len(src_y_nums) == src_row_count \
            and len(dest_y_nums) == dest_row_count:
        pairs = sorted(zip(src_y_nums, src_row_data), key=lambda p: p[0])
        src_y_sorted = [p[0] for p in pairs]
        src_rows_sorted = [p[1] for p in pairs]
        row_strategy = ('numeric', src_y_sorted, dest_y_nums, src_rows_sorted)
    elif src_y_strs and dest_row_labels \
            and len(src_y_strs) == src_row_count:
        idx_map = _nearest_neighbour_label_map(src_y_strs, dest_row_labels)
        row_strategy = ('enum_label', idx_map, src_row_data)
    else:
        src_row_norm = ([i * (dest_row_count - 1) / (src_row_count - 1)
                         for i in range(src_row_count)]
                        if src_row_count >= 2 else [0.0])
        row_strategy = ('index', src_row_norm, src_row_data)

    # Build column strategy
    if x_axis_numeric and src_x_nums and dest_x_nums \
            and len(src_x_nums) == max_src_cols:
        src_x_sorted_pairs = sorted(enumerate(src_x_nums), key=lambda p: p[1])
        src_x_sorted = [p[1] for p in src_x_sorted_pairs]
        col_perm = [p[0] for p in src_x_sorted_pairs]
        col_strategy = ('numeric', src_x_sorted, dest_x_nums, col_perm)
    elif src_x_strs and len(src_x_strs) == max_src_cols:
        col_strategy = ('enum_label', src_x_strs)
    else:
        col_strategy = ('index', None)

    # Resample data
    for row_i, dest_vg in enumerate(dest_vgs):
        dest_v_elems = dest_vg.findall("V", namespaces=namespace)
        dest_col_count = len(dest_v_elems)
        if dest_col_count == 0:
            continue

        intermediate = []
        if row_strategy[0] == 'numeric':
            _, src_y_sorted, dest_y_nums_local, src_rows_sorted = row_strategy
            dest_y_pos = dest_y_nums_local[row_i]
            for col_j in range(max_src_cols):
                col_signal = [
                    r[col_j] if col_j < len(r) else (r[-1] if r else 0.0)
                    for r in src_rows_sorted
                ]
                v = interp_extrap_values(
                    src_y_sorted, col_signal, [dest_y_pos],
                    f"{label} (y-resample col={col_j})")[0]
                intermediate.append(v)
        elif row_strategy[0] == 'enum_label':
            _, idx_map, src_rows = row_strategy
            chosen_row = src_rows[idx_map[row_i]] if idx_map else []
            for col_j in range(max_src_cols):
                intermediate.append(
                    chosen_row[col_j] if col_j < len(chosen_row)
                    else (chosen_row[-1] if chosen_row else 0.0)
                )
        else:
            _, src_row_norm, src_rows = row_strategy
            for col_j in range(max_src_cols):
                col_signal = [
                    r[col_j] if col_j < len(r) else (r[-1] if r else 0.0)
                    for r in src_rows
                ]
                v = interp_extrap_values(
                    src_row_norm, col_signal, [row_i],
                    f"{label} (row-index col={col_j})")[0]
                intermediate.append(v)

        if col_strategy[0] == 'numeric':
            _, src_x_sorted, dest_x_nums_local, col_perm = col_strategy
            intermediate_sorted = [intermediate[j] for j in col_perm]
            final_col_vals = interp_extrap_values(
                src_x_sorted, intermediate_sorted, dest_x_nums_local,
                f"{label} (x-resample row={row_i})")
        elif col_strategy[0] == 'enum_label':
            _, src_x_labels = col_strategy
            idx_map = _nearest_neighbour_label_map(src_x_labels, dest_x_strs)
            if idx_map:
                final_col_vals = [intermediate[i] for i in idx_map]
            else:
                idx_map = _nearest_neighbour_index_map(max_src_cols, dest_col_count)
                final_col_vals = [intermediate[i] for i in idx_map]
        else:
            if max_src_cols >= 2:
                src_col_norm_local = [
                    j * (dest_col_count - 1) / (max_src_cols - 1)
                    for j in range(max_src_cols)
                ]
                final_col_vals = interp_extrap_values(
                    src_col_norm_local, intermediate, list(range(dest_col_count)),
                    f"{label} (col-index row={row_i})")
            else:
                final_col_vals = [intermediate[0]] * dest_col_count

        for col_i, v_elem in enumerate(dest_v_elems):
            v_elem.text = _format_num(final_col_vals[col_i])

        if y_axis_numeric:
            lbl_el = dest_vg.find("LABEL", namespaces=namespace)
            if lbl_el is not None and y_vals_new_display \
                    and row_i < len(y_vals_new_display):
                new_lbl = y_vals_new_display[row_i]
                lbl_el.text = (_format_num(new_lbl) if isinstance(new_lbl, (int, float))
                               else str(new_lbl))

    value_dict = {}
    for vg in dest_vgs:
        lbl_el = vg.find("LABEL", namespaces=namespace)
        key = lbl_el.text.strip() if (lbl_el is not None and lbl_el.text) \
            else str(id(vg))
        value_dict[key] = _parse_floats(vg.findall("V", namespaces=namespace))

    return 'updated', {'label': label, 'values': value_dict,
                       'x_dim': len(x_vals_new_display),
                       'y_dim': len(y_vals_new_display),
                       'x_dim_val': x_vals_new_display,
                       'y_dim_val': y_vals_new_display,
                       'status': 'Updated'}


def _update_curve_1d_json(label, src_entry, dest_sw, namespace):
    src_z = src_entry.get('value', [])
    src_axes = src_entry.get('axes', [])
    if not isinstance(src_z, list) or not src_z:
        file_logger.warning(f"  {label}: curve value missing or not a list — skipping")
        return 'skip', None
    if not isinstance(src_axes, list):
        src_axes = []

    axis_dest = dest_sw.findall(".//SW-VALUES-PHYS", namespaces=namespace)
    if not axis_dest:
        return 'skip', None

    dest_z_elems = axis_dest[0].findall("V", namespaces=namespace)
    if not dest_z_elems:
        dest_z_elems = axis_dest[0].findall("VT", namespaces=namespace)

    src_z_n = len(src_z)
    dest_z_n = len(dest_z_elems)
    if src_z_n == 0 or dest_z_n == 0:
        return 'skip', None

    src_x = _safe_float_list(_axis_values(src_axes[0])) if src_axes else []
    dest_x_elems = []
    if len(axis_dest) > 1:
        dest_x_elems = axis_dest[1].findall("V", namespaces=namespace)
        if not dest_x_elems:
            dest_x_elems = axis_dest[1].findall("VT", namespaces=namespace)
    dest_x = _parse_floats(dest_x_elems)

    try:
        src_z_float = [float(v) for v in src_z]
        numeric_z = True
    except (ValueError, TypeError):
        numeric_z = False
    numeric_x = bool(src_x and dest_x)

    old_z_texts = [e.text or "" for e in dest_z_elems]

    if numeric_z and numeric_x and len(src_x) == src_z_n:
        new_z = interp_extrap_values(
            src_x, src_z_float, dest_x, f"{label} (z@dest_x)")
        new_z_texts = [_format_num(v) for v in new_z]
    elif numeric_z and numeric_x and len(src_x) != src_z_n:
        file_logger.warning(
            f"  {label}: src z length ({src_z_n}) != src x length ({len(src_x)}) "
            f"— falling back to index-based interpolation."
        )
        src_x_norm = ([i * (dest_z_n - 1) / (src_z_n - 1) for i in range(src_z_n)]
                      if src_z_n >= 2 else [0.0])
        new_z = interp_extrap_values(
            src_x_norm, src_z_float, list(range(dest_z_n)), f"{label} (index fallback)")
        new_z_texts = [_format_num(v) for v in new_z]
    elif numeric_z:
        src_x_norm = ([i * (dest_z_n - 1) / (src_z_n - 1) for i in range(src_z_n)]
                      if src_z_n >= 2 else [0.0])
        new_z = interp_extrap_values(
            src_x_norm, src_z_float, list(range(dest_z_n)), f"{label} (index)")
        new_z_texts = [_format_num(v) for v in new_z]
    else:
        idx_map = _nearest_neighbour_index_map(src_z_n, dest_z_n)
        new_z_texts = [str(src_z[i]) for i in idx_map]

    if [t.strip() for t in old_z_texts] == [t.strip() for t in new_z_texts]:
        return 'no_change', None

    for elem, text in zip(dest_z_elems, new_z_texts):
        elem.text = text

    x_vals = [e.text for e in dest_x_elems] if dest_x_elems else []
    z_vals = [e.text for e in dest_z_elems]
    return 'updated', {'label': label, 'x_values': x_vals,
                       'z_values': z_vals, 'x_dim': len(x_vals),
                       'status': 'Updated'}


# ─────────────────────────────────────────────────────────────────────────────
# MAIN UPDATE FUNCTION — CDFX source → CDFX destination
# ─────────────────────────────────────────────────────────────────────────────

def override_incorrect_values(correct_cdfx_file, incorrect_cdfx_file, namespace):

    if not os.path.exists(incorrect_cdfx_file):
        try:
            shutil.copyfile(correct_cdfx_file, incorrect_cdfx_file)
            print(f"Created: {incorrect_cdfx_file}")
            return pd.DataFrame({'label': [], 'value': [],
                                 'status': ['Destination Created by Copying Source']})
        except Exception as e:
            logging.error(f"Failed to copy source to destination: {e}")
            return pd.DataFrame()

    correct_df = extract_labels_with_c(correct_cdfx_file, namespace)
    incorrect_df = extract_labels_with_c(incorrect_cdfx_file, namespace)

    if correct_df.empty:
        logging.error("No data extracted from the source file. Aborting update.")
        return pd.DataFrame()

    total_correct_labels = len(correct_df)
    total_destination_labels = len(incorrect_df)
    source_labels = set(correct_df['label'])
    dest_labels = set(incorrect_df['label'])
    dest_only_count = len(dest_labels - source_labels)

    print(f"Source: {total_correct_labels} labels | "
          f"Destination: {total_destination_labels} labels | "
          f"Dest-only: {dest_only_count} | Log: calibration_process.log")

    merged_df = pd.merge(correct_df, incorrect_df, on='label',
                         suffixes=('_correct', '_incorrect'),
                         how='left', indicator=True)

    try:
        tree_incorrect = ET.parse(incorrect_cdfx_file)
        root_incorrect = tree_incorrect.getroot()
    except Exception as e:
        logging.error(f"Failed to parse destination CDFX file: {e}")
        return pd.DataFrame()

    try:
        tree_correct = ET.parse(correct_cdfx_file)
        root_correct = tree_correct.getroot()
    except Exception as e:
        logging.error(f"Failed to parse source CDFX file: {e}")
        return pd.DataFrame()

    label_index_correct = build_label_index(root_correct, namespace)
    label_index_incorrect = build_label_index(root_incorrect, namespace)

    updated_data = []
    start_time = time.time()
    last_progress_time = start_time
    progress_interval = 120
    processed_count = 0
    updated_count = 0
    no_update_needed_count = 0
    unhandled_suffix_count = 0
    unhandled_suffixes_seen = {}

    for idx, row in merged_df.iterrows():
        label = row['label']
        correct_sw = label_index_correct.get(label)
        incorrect_sw = label_index_incorrect.get(label)

        if row['_merge'] == 'left_only':
            continue
        if correct_sw is None or incorrect_sw is None:
            continue

        processed_count += 1

        current_time = time.time()
        if current_time - last_progress_time >= progress_interval:
            elapsed_min = (current_time - start_time) / 60
            pct = 100 * processed_count / total_destination_labels
            print(f"[{datetime.now().strftime('%H:%M:%S')}] {elapsed_min:.0f}min | "
                  f"{processed_count}/{total_destination_labels} ({pct:.1f}%) | "
                  f"Updated: {updated_count} | No change: {no_update_needed_count}")
            last_progress_time = current_time

        if label.endswith(('_C', '_CW', '_c')):
            status, record = _update_scalar_cdfx(
                label, correct_sw, incorrect_sw, row, namespace)
        elif label.endswith('_CA'):
            status, record = _update_array_1d_cdfx(
                label, correct_sw, incorrect_sw, namespace)
        elif label.endswith(('_MAP', '_M')):
            status, record = _update_map_2d_cdfx(
                label, correct_sw, incorrect_sw, namespace)
        elif label.endswith(('_T', '_CUR', '_Cur')):
            status, record = _update_curve_1d_cdfx(
                label, correct_sw, incorrect_sw, namespace)
        else:
            unhandled_suffix_count += 1
            suffix = label.rsplit('_', 1)[-1] if '_' in label else label
            unhandled_suffixes_seen[suffix] = (
                unhandled_suffixes_seen.get(suffix, 0) + 1)
            file_logger.info(
                f"  Unhandled suffix: {label} (suffix=_{suffix}) — skipped"
            )
            continue

        if status == 'updated':
            updated_data.append(record)
            updated_count += 1
        elif status == 'no_change':
            no_update_needed_count += 1

    try:
        tree_incorrect.write(incorrect_cdfx_file, encoding='utf-8',
                             xml_declaration=True)
        print(f"\nDestination file updated and saved to '{incorrect_cdfx_file}'.")
    except Exception as e:
        logging.error(f"Failed to write updates: {e}")
        return pd.DataFrame()

    _print_summary(updated_count, no_update_needed_count, total_destination_labels,
                   dest_labels - source_labels, time.time() - start_time,
                   unhandled_suffix_count, unhandled_suffixes_seen)
    return pd.DataFrame(updated_data)


# ─────────────────────────────────────────────────────────────────────────────
# MAIN UPDATE FUNCTION — JSON source → CDFX destination
# ─────────────────────────────────────────────────────────────────────────────

def override_incorrect_values_json_src(json_src_file, cdfx_dest_file, namespace):
    try:
        with open(json_src_file, 'r') as f:
            json_data = json.load(f)
    except FileNotFoundError:
        logging.error(f"JSON source not found: '{json_src_file}'")
        return pd.DataFrame()
    except json.JSONDecodeError as e:
        logging.error(f"JSON decode error in '{json_src_file}': {e}")
        return pd.DataFrame()

    try:
        tree_dest = ET.parse(cdfx_dest_file)
        root_dest = tree_dest.getroot()
    except Exception as e:
        logging.error(f"Failed to parse destination CDFX '{cdfx_dest_file}': {e}")
        return pd.DataFrame()

    label_index_dest = build_label_index(root_dest, namespace)

    source_labels = set(json_data.keys())
    dest_labels = set(label_index_dest.keys())
    total_dest = len(dest_labels)
    dest_only = len(dest_labels - source_labels)

    print(f"Source (JSON): {len(source_labels)} labels | "
          f"Destination (CDFX): {total_dest} labels | "
          f"Dest-only: {dest_only} | Log: calibration_process.log")

    updated_data = []
    updated_count = 0
    no_update_needed_count = 0
    processed_count = 0
    unhandled_suffix_count = 0
    unhandled_suffixes_seen = {}
    start_time = time.time()
    last_progress_time = start_time
    progress_interval = 120

    for label in dest_labels:
        if label not in source_labels:
            continue

        dest_sw = label_index_dest[label]
        src_entry = json_data[label]
        if not isinstance(src_entry, dict):
            file_logger.warning(
                f"  {label}: JSON entry must be a dict, got "
                f"{type(src_entry).__name__} — skipping"
            )
            continue

        processed_count += 1

        current_time = time.time()
        if current_time - last_progress_time >= progress_interval:
            elapsed_min = (current_time - start_time) / 60
            pct = 100 * processed_count / total_dest
            print(f"[{datetime.now().strftime('%H:%M:%S')}] {elapsed_min:.0f}min | "
                  f"{processed_count}/{total_dest} ({pct:.1f}%) | "
                  f"Updated: {updated_count} | No change: {no_update_needed_count}")
            last_progress_time = current_time

        if label.endswith(('_C', '_CW', '_c')):
            status, record = _update_scalar_json(label, src_entry, dest_sw, namespace)
        elif label.endswith('_CA'):
            status, record = _update_array_1d_json(label, src_entry, dest_sw, namespace)
        elif label.endswith(('_MAP', '_M')):
            status, record = _update_map_2d_json(label, src_entry, dest_sw, namespace)
        elif label.endswith(('_T', '_CUR', '_Cur')):
            status, record = _update_curve_1d_json(label, src_entry, dest_sw, namespace)
        else:
            unhandled_suffix_count += 1
            suffix = label.rsplit('_', 1)[-1] if '_' in label else label
            unhandled_suffixes_seen[suffix] = (
                unhandled_suffixes_seen.get(suffix, 0) + 1)
            file_logger.info(
                f"  Unhandled suffix: {label} (suffix=_{suffix}) — skipped"
            )
            continue

        if status == 'updated':
            updated_data.append(record)
            updated_count += 1
        elif status == 'no_change':
            no_update_needed_count += 1

    try:
        tree_dest.write(cdfx_dest_file, encoding='utf-8', xml_declaration=True)
        print(f"\nDestination file updated and saved to '{cdfx_dest_file}'.")
    except Exception as e:
        logging.error(f"Failed to write updates to '{cdfx_dest_file}': {e}")
        return pd.DataFrame()

    _print_summary(updated_count, no_update_needed_count, total_dest,
                   dest_labels - source_labels, time.time() - start_time,
                   unhandled_suffix_count, unhandled_suffixes_seen)
    return pd.DataFrame(updated_data)


# ─────────────────────────────────────────────────────────────────────────────
# SHARED SUMMARY PRINTER
# ─────────────────────────────────────────────────────────────────────────────

def _print_summary(updated_count, no_update_needed_count, total_dest,
                   dest_only_set, elapsed_seconds,
                   unhandled_count=0, unhandled_seen=None):
    dest_only_count = len(dest_only_set)
    pct_updated = 100 * updated_count / total_dest if total_dest else 0
    pct_no_change = 100 * no_update_needed_count / total_dest if total_dest else 0
    attention = total_dest - updated_count - no_update_needed_count
    pct_dest_only = 100 * dest_only_count / total_dest if total_dest else 0
    total_min = elapsed_seconds / 60

    print(f"\n{'='*60}")
    print(f"SUMMARY:")
    print(f"  Updated:        {updated_count:5d} ({pct_updated:.1f}%)")
    print(f"  No change:      {no_update_needed_count:5d} ({pct_no_change:.1f}%)")
    print(f"  Need attention: {attention:5d} "
          f"({100*attention/total_dest if total_dest else 0:.1f}%)")
    print(f"  Dest-only:      {dest_only_count:5d} ({pct_dest_only:.1f}%)")
    if unhandled_count:
        print(f"  Unhandled:      {unhandled_count:5d} "
              f"(suffixes: {unhandled_seen})")
    print(f"  Total:          {total_dest:5d}")
    print(f"  Time:           {total_min:.1f} min")
    print(f"{'='*60}")


# ─────────────────────────────────────────────────────────────────────────────
# ENTRY POINT
# ─────────────────────────────────────────────────────────────────────────────

if __name__ == "__main__":

    namespace = {'autosar': 'http://autosar.org/schema/r4.0'}

    # Source files can be any mix of .CDFX and .json.
    source_files = ['ADM_65.CDFX', '40cdfx.json']
    destination_file = 'ADM_28.CDFX'

    all_labels_updated = set()
    all_labels = set()
    summary = []

    dest_df = extract_labels_with_c(destination_file, namespace)
    if dest_df.empty:
        print("No labels found in destination file. Exiting.")
        exit(1)

    all_labels = set(dest_df['label'])
    labels_left = set(all_labels)
    print(f"Initial: {len(all_labels)} labels in destination.")

    for idx, src in enumerate(source_files):
        print(f"\n--- Processing Source {idx+1}: {src} ---")
        temp_dest = f"_temp_dest_{idx}.CDFX"

        filter_destination_file(destination_file, temp_dest, labels_left, namespace)

        if _file_type(src) == 'json':
            try:
                with open(src, 'r') as f:
                    src_labels = set(json.load(f).keys())
            except Exception as e:
                logging.error(f"Could not read JSON source '{src}': {e}")
                src_labels = set()
            updated_df = override_incorrect_values_json_src(src, temp_dest, namespace)
        else:
            src_df = extract_labels_with_c(src, namespace)
            src_labels = set(src_df['label']) if not src_df.empty else set()
            updated_df = override_incorrect_values(src, temp_dest, namespace)

        updated_labels = set(updated_df['label']) if not updated_df.empty else set()
        all_labels_updated.update(updated_labels)

        found_in_src = labels_left & src_labels
        found_but_unchanged = found_in_src - updated_labels
        not_found_in_src = labels_left - found_in_src

        merge_updated_instances(destination_file, temp_dest, updated_labels, namespace)
        os.remove(temp_dest)

        labels_left = not_found_in_src

        print(f"Source {idx+1} summary:")
        print(f"  Labels considered this round: {len(found_in_src) + len(not_found_in_src)}")
        print(f"    Updated:              {len(updated_labels)}")
        print(f"    Found but unchanged:  {len(found_but_unchanged)}")
        print(f"    Still need attention: {len(not_found_in_src)}")
        summary.append({'source': src, 'updated': len(updated_labels),
                        'found_but_unchanged': len(found_but_unchanged),
                        'still_need_attention': len(not_found_in_src),
                        'remaining': len(labels_left)})

    print("\n=== FINAL SUMMARY ===")
    for idx, s in enumerate(summary):
        print(f"Source {idx+1} ({s['source']}): Updated {s['updated']}, "
              f"Found but unchanged: {s['found_but_unchanged']}, "
              f"Still need attention: {s['still_need_attention']}, "
              f"Remaining after: {s['remaining']}")

    pct_left = 100 * len(labels_left) / len(all_labels) if all_labels else 0
    print(f"\nTotal labels in destination:      {len(all_labels)}")
    print(f"Total updated from all sources:   {len(all_labels_updated)}")
    print(f"Labels left after all sources:    {len(labels_left)} ({pct_left:.1f}%)")

    if labels_left:
        pd.DataFrame({'label': sorted(labels_left)}).to_excel(
            'needs_attention_labels.xlsx', index=False)
        print("Labels needing attention saved to needs_attention_labels.xlsx.")

Initial: 86779 labels in destination.

--- Processing Source 1: ADM_65.CDFX ---
Source: 57725 labels | Destination: 86779 labels | Dest-only: 40292 | Log: calibration_process.log



Destination file updated and saved to '_temp_dest_0.CDFX'.

SUMMARY:
  Updated:         6991 (8.1%)
  No change:      39458 (45.5%)
  Need attention: 40330 (46.5%)
  Dest-only:      40292 (46.4%)
  Total:          86779
  Time:           0.1 min
Source 1 summary:
  Labels considered this round: 86779
    Updated:              6991
    Found but unchanged:  39496
    Still need attention: 40292

--- Processing Source 2: ADM_40.CDFX ---
Source: 79865 labels | Destination: 40292 labels | Dest-only: 17379 | Log: calibration_process.log



Destination file updated and saved to '_temp_dest_1.CDFX'.

SUMMARY:
  Updated:        19212 (47.7%)
  No change:       3696 (9.2%)
  Need attention: 17384 (43.1%)
  Dest-only:      17379 (43.1%)
  Total:          40292
  Time:           0.1 min
Source 2 summary:
  Labels considered this round: 40292
    Updated:              19212
    Found but unchanged:  3701
    Still need attention: 17379

=== FINAL SUMMARY ===
Source 1 (ADM_65.CDFX): Updated 6991, Found but unchanged: 39496, Still need attention: 40292, Remaining after: 40292
Source 2 (ADM_40.CDFX): Updated 19212, Found but unchanged: 3701, Still need attention: 17379, Remaining after: 17379

Total labels in destination:      86779
Total updated from all sources:   26203
Labels left after all sources:    17379 (20.0%)
Labels needing attention saved to needs_attention_labels.xlsx.
